# ch14_state_space

*Computers in the Loop: Digital Control from Sampling to Networked Agents*: EECE 5610 lecture notes prepared and revised by Milad Siami.

Chapters 1–11 of the notes are based on the lecture notes of Prof. Mario Sznaier; all credits are in `ATTRIBUTION.md`.



Run the cells from top to bottom (in Colab: *Runtime > Run all*). Figures are also saved to a `figures` folder.


In [ ]:
#!/usr/bin/env python3
"""EECE 5610, Chapter 14 companion: state-space design for a sampled plant.

The running example is a ZOH followed by G_p(s) = 1/(s(s+1)), sampled at T = 1 s,
with states x1 = c (position) and x2 = dc/dt (velocity).

What this script does, in the order of the chapter:
  1. samples the plant exactly: Phi = e^{AT}, Gamma = (int_0^T e^{A s} ds) B;
  2. checks controllability and observability with the rank tests;
  3. places the poles at 0.5 +- j0.5 (Ackermann), and designs a deadbeat gain;
  4. adds the reference gain Nbar, and integral action by state augmentation;
  5. designs prediction and current observers with poles at 0.2, 0.2;
  6. forms the controller-observer D(z) and checks the closed-loop poles;
  7. computes the discrete LQR gain by iterating the Riccati equation.

Each step prints its numbers; the figures go to ./ch14_figures/ next to this file.
Needs numpy, scipy and matplotlib.

Run:  python3 ch14_state_space.py
"""

In [ ]:
import os

import numpy as np
import scipy.linalg as sla
import matplotlib

import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
HERE = os.getcwd()
FIGDIR = os.path.join(HERE, "ch14_figures")
os.makedirs(FIGDIR, exist_ok=True)

In [ ]:
# ---------------------------------------------------------------------------
# helpers
# ---------------------------------------------------------------------------
def sample_plant(A, B, T):
    """Exact ZOH sampling. The matrix exponential of [[A, B], [0, 0]] * T
    contains Phi in its top-left block and Gamma in its top-right block."""
    n, nu = B.shape
    M = np.zeros((n + nu, n + nu))
    M[:n, :n] = A
    M[:n, n:] = B
    E = sla.expm(M * T)
    return E[:n, :n], E[:n, n:]

In [ ]:
def ctrb(Phi, Gam):
    """Controllability matrix [Gam, Phi Gam, ..., Phi^(n-1) Gam]."""
    n = Phi.shape[0]
    return np.hstack([np.linalg.matrix_power(Phi, i) @ Gam for i in range(n)])

In [ ]:
def obsv(Phi, C):
    """Observability matrix [C; C Phi; ...; C Phi^(n-1)]."""
    n = Phi.shape[0]
    return np.vstack([C @ np.linalg.matrix_power(Phi, i) for i in range(n)])

In [ ]:
def poly_of_matrix(coef, M):
    """alpha(M) = M^n + a1 M^(n-1) + ... + an I, with coef = [1, a1, ..., an]."""
    n = len(coef) - 1
    return sum(c * np.linalg.matrix_power(M, n - i) for i, c in enumerate(coef))

In [ ]:
def ackermann(Phi, Gam, coef):
    """State-feedback gain K (a row) with det(zI - Phi + Gam K) = alpha(z)."""
    n = Phi.shape[0]
    last_row = np.zeros((1, n))
    last_row[0, -1] = 1.0
    return last_row @ np.linalg.solve(ctrb(Phi, Gam), poly_of_matrix(coef, Phi))

In [ ]:
def observer_gain(Phi, C, coef):
    """Prediction-observer gain L (a column) with det(zI - Phi + L C) = alpha(z).
    Duality: it is the transpose of Ackermann's gain for the pair (Phi^T, C^T)."""
    return ackermann(Phi.T, C.T, coef).T

In [ ]:
def show(name, value):
    print(f"{name:>28s} = {value}")

In [ ]:
# ---------------------------------------------------------------------------
# 1. the plant and its exact sampled model
# ---------------------------------------------------------------------------
A = np.array([[0.0, 1.0], [0.0, -1.0]])

In [ ]:
B = np.array([[0.0], [1.0]])
C = np.array([[1.0, 0.0]])
T = 1.0
Phi, Gam = sample_plant(A, B, T)
a = np.exp(-T)

print("1. Sampled model")
show("Phi", Phi.round(4).tolist())
show("Gamma", Gam.ravel().round(4).tolist())

In [ ]:
# compare with the closed forms found by hand
assert np.allclose(Phi, [[1, 1 - a], [0, a]]) and np.allclose(Gam, [[T - 1 + a], [1 - a]])

In [ ]:
# pulse transfer function G(z) = C (zI - Phi)^{-1} Gamma at one test point
z0 = 0.3 + 0.7j

In [ ]:
G_state = (C @ np.linalg.solve(z0 * np.eye(2) - Phi, Gam))[0, 0]
G_table = (a * z0 + 1 - 2 * a) / ((z0 - 1) * (z0 - a))
show("G(z0) two ways", f"{G_state:.6f}  {G_table:.6f}")

In [ ]:
# ---------------------------------------------------------------------------
# 2. controllability and observability
# ---------------------------------------------------------------------------
print("\n2. Rank tests")

In [ ]:
show("det Ctrb", round(np.linalg.det(ctrb(Phi, Gam)), 4))
show("det Obsv", round(np.linalg.det(obsv(Phi, C)), 4))

In [ ]:
# the oscillator 1/(s^2 + 1) sampled at T = pi loses both
Ao = np.array([[0.0, 1.0], [-1.0, 0.0]])

In [ ]:
for To in [np.pi / 2, np.pi]:
    Po, Go = sample_plant(Ao, B, To)
    show(f"oscillator rank Ctrb, T={To:.3f}", np.linalg.matrix_rank(ctrb(Po, Go), tol=1e-9))

In [ ]:
# ---------------------------------------------------------------------------
# 3. pole placement and deadbeat
# ---------------------------------------------------------------------------
print("\n3. State feedback")

In [ ]:
alpha_c = [1.0, -1.0, 0.5]  # (z - 0.5 - 0.5j)(z - 0.5 + 0.5j)
K = ackermann(Phi, Gam, alpha_c)
show("K (poles 0.5 +- j0.5)", K.ravel().round(4).tolist())
show("eig(Phi - Gamma K)", np.linalg.eigvals(Phi - Gam @ K).round(4).tolist())
Kdb = ackermann(Phi, Gam, [1.0, 0.0, 0.0])
show("deadbeat K", Kdb.ravel().round(4).tolist())
show("(Phi - Gamma Kdb)^2", np.abs((Phi - Gam @ Kdb) @ (Phi - Gam @ Kdb)).round(12).tolist())

In [ ]:
def regulate(Kgain, x0, steps):
    """Run x(k+1) = (Phi - Gamma K) x(k); also return c(t) between samples.
    Inside one period the input is constant, so the exact solution is
    c(kT + tau) = x1 + x2 (1 - e^{-tau}) + m (tau - 1 + e^{-tau})."""
    x = np.array(x0, dtype=float)
    t_fine, c_fine, m_seq = [], [], []
    for k in range(steps):
        m = -(Kgain @ x)[0]
        m_seq.append(m)
        for tau in np.linspace(0, T, 21)[:-1]:
            t_fine.append(k * T + tau)
            c_fine.append(x[0] + x[1] * (1 - np.exp(-tau)) + m * (tau - 1 + np.exp(-tau)))
        x = Phi @ x + Gam[:, 0] * m
    t_fine.append(steps * T)
    c_fine.append(x[0])
    return np.array(t_fine), np.array(c_fine), np.array(m_seq)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))

In [ ]:
for Kgain, lab in [(K, "poles 0.5 +- j0.5"), (Kdb, "deadbeat")]:
    t, c, m = regulate(Kgain, [1.0, 0.0], 8)
    ax[0].plot(t, c, label=lab)
    ax[0].plot(t[::20], c[::20], "o", ms=3, color=ax[0].lines[-1].get_color())
    ax[1].step(np.arange(9), np.append(m, m[-1]), where="post", label=lab)

In [ ]:
ax[0].set(xlabel="t (s)", ylabel="c(t)", title="regulation from x(0) = [1, 0]")
ax[1].set(xlabel="t (s)", ylabel="m(t)", title="control (held by the ZOH)")

In [ ]:
for a_ in ax:
    a_.grid(alpha=0.3)
    a_.legend()

In [ ]:
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "ch14_pole_placement.png"), dpi=150)
plt.close(fig)

In [ ]:
# deadbeat effort grows like 1/T^2
print("   deadbeat effort for several T (x(0) = [1, 0]):")

In [ ]:
for Ts in [2.0, 1.0, 0.5, 0.2, 0.1]:
    P_, G_ = sample_plant(A, B, Ts)
    Kt = ackermann(P_, G_, [1.0, 0.0, 0.0])
    print(f"      T = {Ts:4.1f}   K = {Kt.ravel().round(3)}   m(0) = {-Kt[0, 0]:9.3f}")

In [ ]:
# ---------------------------------------------------------------------------
# 4. reference gain and integral action
# ---------------------------------------------------------------------------
print("\n4. Tracking")

In [ ]:
Nbar = 1.0 / (C @ np.linalg.solve(np.eye(2) - Phi + Gam @ K, Gam))[0, 0]
show("Nbar", round(Nbar, 4))

In [ ]:
# augmented model: x_I(k+1) = x_I(k) + r(k) - c(k)
Phi_a = np.block([[Phi, np.zeros((2, 1))], [-C, np.ones((1, 1))]])

In [ ]:
Gam_a = np.vstack([Gam, [[0.0]]])
Ka = ackermann(Phi_a, Gam_a, np.polymul(alpha_c, [1.0, -0.5]))
k1, k2, kI = Ka[0, 0], Ka[0, 1], -Ka[0, 2]
show("[k1, k2, kI]", [round(float(k1), 4), round(float(k2), 4), round(float(kI), 4)])

N = 31
w = np.where(np.arange(N) >= 15, 0.2, 0.0)  # constant input disturbance from k = 15
c_nbar, c_int = np.zeros(N), np.zeros(N)
x = np.zeros(2)

In [ ]:
for k in range(N):
    c_nbar[k] = x[0]
    m = -(K @ x)[0] + Nbar * 1.0
    x = Phi @ x + Gam[:, 0] * (m + w[k])

In [ ]:
xa = np.zeros(3)

In [ ]:
for k in range(N):
    c_int[k] = xa[0]
    m = -(Ka @ xa)[0]
    xa = Phi_a @ xa + Gam_a[:, 0] * (m + w[k]) + np.array([0.0, 0.0, 1.0])

In [ ]:
show("final c, Nbar only", round(c_nbar[-1], 3))
show("final c, integral action", round(c_int[-1], 3))

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(c_nbar, "o-", ms=3, label="Nbar only")
ax.plot(c_int, "s-", ms=3, label="integral action")
ax.axhline(1.0, color="gray", lw=0.8, ls=":")
ax.axvline(15, color="gray", lw=0.8, ls="--")
ax.set(xlabel="k", ylabel="c(k)", title="unit step; +0.2 added at the plant input from k = 15")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "ch14_integral_action.png"), dpi=150)
plt.close(fig)

In [ ]:
# ---------------------------------------------------------------------------
# 5. observers
# ---------------------------------------------------------------------------
print("\n5. Observers, poles at 0.2, 0.2")

In [ ]:
alpha_o = [1.0, -0.4, 0.04]
L = observer_gain(Phi, C, alpha_o)
Lc = observer_gain(Phi, C @ Phi, alpha_o)  # current observer: C Phi in place of C
show("prediction L", L.ravel().round(4).tolist())
show("current Lc", Lc.ravel().round(4).tolist())
show("eig(Phi - L C)", np.linalg.eigvals(Phi - L @ C).real.round(4).tolist())
show("eig(Phi - Lc C Phi)", np.linalg.eigvals(Phi - Lc @ C @ Phi).real.round(4).tolist())

In [ ]:
# separation: feed back the estimate, compare with true-state feedback
steps = 11

In [ ]:
x, xh, xs = np.array([1.0, 0.0]), np.zeros(2), np.array([1.0, 0.0])
c_obs, c_sf, xh1 = [], [], []

In [ ]:
for k in range(steps):
    c_obs.append(x[0])
    c_sf.append(xs[0])
    xh1.append(xh[0])
    m = -(K @ xh)[0]
    m_sf = -(K @ xs)[0]
    c_meas = (C @ x)[0]
    xh = Phi @ xh + Gam[:, 0] * m + L[:, 0] * (c_meas - (C @ xh)[0])
    x = Phi @ x + Gam[:, 0] * m
    xs = Phi @ xs + Gam[:, 0] * m_sf

In [ ]:
big = np.block([[Phi, -Gam @ K], [L @ C, Phi - Gam @ K - L @ C]])
show("closed-loop poles (x, xhat)", np.sort_complex(np.linalg.eigvals(big)).round(4).tolist())

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(c_sf, "o-", ms=3, label="c(k), m = -K x")
ax.plot(c_obs, "s-", ms=3, label="c(k), m = -K xhat")
ax.plot(xh1, "^--", ms=3, label="estimate xhat1(k)")
ax.set(xlabel="k", title="observer-based regulator, xhat(0) = 0")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "ch14_observer.png"), dpi=150)
plt.close(fig)

In [ ]:
# ---------------------------------------------------------------------------
# 6. the controller-observer as a transfer function D(z)
# ---------------------------------------------------------------------------
print("\n6. D(z) = K (zI - Phi + Gamma K + L C)^{-1} L")

In [ ]:
F = Phi - Gam @ K - L @ C
den_D = np.poly(F)  # det(zI - F)

In [ ]:
# numerator: D(z) * det(zI - F) is a first-degree polynomial; fit it from two points
zz = np.array([0.0, 1.0])

In [ ]:
vals = [(K @ np.linalg.solve(z * np.eye(2) - F, L))[0, 0] * np.polyval(den_D, z) for z in zz]
num_D = np.array([vals[1] - vals[0], vals[0]])
show("numerator of D", num_D.round(4).tolist())
show("denominator of D", den_D.round(4).tolist())
num_G, den_G = np.array([a, 1 - 2 * a]), np.poly([1.0, a])
char = np.polyadd(np.polymul(den_G, den_D), np.polymul(num_G, num_D))
show("1 + D G = 0, polynomial", char.round(4).tolist())
show("roots", np.sort_complex(np.roots(char)).round(4).tolist())

In [ ]:
# ---------------------------------------------------------------------------
# 7. discrete LQR
# ---------------------------------------------------------------------------
print("\n7. LQR, Q = C^T C")

In [ ]:
Q = C.T @ C

In [ ]:
def lqr_by_iteration(rho, iters=200):
    """Iterate P <- Q + Phi'P Phi - Phi'P Gam (rho + Gam'P Gam)^{-1} Gam'P Phi from P = 0."""
    P = np.zeros((2, 2))
    for _ in range(iters):
        s = rho + (Gam.T @ P @ Gam)[0, 0]
        P = Q + Phi.T @ P @ Phi - (Phi.T @ P @ Gam) @ (Gam.T @ P @ Phi) / s
    Kl = (Gam.T @ P @ Phi) / (rho + (Gam.T @ P @ Gam)[0, 0])
    return P, Kl

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))

In [ ]:
for rho in [10.0, 1.0, 0.1]:
    P, Kl = lqr_by_iteration(rho)
    P_are = sla.solve_discrete_are(Phi, Gam, Q, np.array([[rho]]))
    assert np.allclose(P, P_are)  # iteration and the direct solver agree
    poles = np.linalg.eigvals(Phi - Gam @ Kl)
    print(f"   rho = {rho:4.1f}:  K = {Kl.ravel().round(4)}   poles = {poles.round(4)}   |z| = {abs(poles).round(4)}")
    x = np.array([1.0, 0.0])
    cs, ms = [], []
    for k in range(11):
        cs.append(x[0])
        ms.append(-(Kl @ x)[0])
        x = (Phi - Gam @ Kl) @ x
    ax[0].plot(cs, "o-", ms=3, label=f"rho = {rho:g}")
    ax[1].step(range(11), ms, where="post", label=f"rho = {rho:g}")

In [ ]:
ax[0].set(xlabel="k", ylabel="c(k)", title="LQR regulation from x(0) = [1, 0]")
ax[1].set(xlabel="t (s)", ylabel="m(t)", title="LQR control")

In [ ]:
for a_ in ax:
    a_.grid(alpha=0.3)
    a_.legend()

In [ ]:
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "ch14_lqr.png"), dpi=150)
plt.close(fig)

print(f"\nFigures written to {FIGDIR}")

## Explore




In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
r"""**Tune the LQR weight.** The running plant, a zero-order hold and $1/(s(s+1))$, with state
$x = (c, \dot c)$ and state feedback $m(k) = -K x(k)$. The LQR gain minimizes
$\sum_k \big(c(k)^2 + \rho\, m(k)^2\big)$. Start from $x(0) = (1, 0)$. Lower $\rho$ (cheap control)
and the poles move toward the origin: faster, with a bigger first kick $|m(0)|$. Raise $\rho$ and
the controller gets gentle and slow. At $T = 1$ s, $\rho = 1$ gives $K = [0.7007\ \ 0.5891]$ and
poles $0.3688 \pm j0.2111$. For comparison, the gray curve places the poles at $0.5 \pm j0.5$
for every $T$.
"""
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"
T_END = 10.0     # seconds shown


def _plant(T):
    p = np.exp(-T)
    return np.array([[1.0, 1 - p], [0.0, p]]), np.array([[T - 1 + p], [1 - p]])


def _lqr(Phi, Gam, rho):
    """Iterate the Riccati equation from P = 0 until it settles; return K and P."""
    Q = np.array([[1.0, 0.0], [0.0, 0.0]])
    P = np.zeros((2, 2))
    for _ in range(5000):
        s = rho + (Gam.T @ P @ Gam)[0, 0]
        P_new = Q + Phi.T @ P @ Phi - (Phi.T @ P @ Gam) @ (Gam.T @ P @ Phi) / s
        if np.max(np.abs(P_new - P)) < 1e-13:
            P = P_new
            break
        P = P_new
    K = (Gam.T @ P @ Phi) / (rho + (Gam.T @ P @ Gam)[0, 0])
    return K, P


def _ackermann(Phi, Gam, a1, a0):
    """K with det(zI - Phi + Gam K) = z^2 + a1 z + a0."""
    ctrb = np.hstack([Gam, Phi @ Gam])
    alpha = Phi @ Phi + a1 * Phi + a0 * np.eye(2)
    return np.array([[0.0, 1.0]]) @ np.linalg.solve(ctrb, alpha)


def _run(K, Phi, Gam, T, n):
    """Regulate from x(0) = (1, 0); c(t) between the samples is exact (input held)."""
    p = Phi[1, 1]
    tau = np.linspace(0, T, 21)[:-1]
    x = np.array([1.0, 0.0])
    tf, cf, ms = [], [], []
    for k in range(n):
        m = -(K @ x)[0]
        ms.append(m)
        tf.append(k * T + tau)
        cf.append(x[0] + x[1] * (1 - np.exp(-tau)) + m * (tau - 1 + np.exp(-tau)))
        x = Phi @ x + Gam[:, 0] * m
    return np.concatenate(tf), np.concatenate(cf), np.array(ms)


def explore(log10_rho=0.0, T=1.0):
    rho = 10.0 ** log10_rho
    Phi, Gam = _plant(T)
    K, P = _lqr(Phi, Gam, rho)
    poles = np.linalg.eigvals(Phi - Gam @ K)
    K_pp = _ackermann(Phi, Gam, -1.0, 0.5)             # poles 0.5 +- j0.5
    n = int(round(T_END / T))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.0))
    for gain, col, lab in [(K_pp, GRAY, "pole placement 0.5 $\\pm$ j0.5"),
                           (K, ORANGE, f"LQR, $\\rho$ = {rho:.3g}")]:
        tf, cf, ms = _run(gain, Phi, Gam, T, n)
        ax1.plot(tf, cf, color=col, lw=1.8 if col == ORANGE else 1.4, label=lab)
        ax1.plot(np.arange(n) * T, cf[::20], "o", color=col, ms=5 if T >= 0.3 else 3)
        ax2.step(np.arange(n + 1) * T, np.append(ms, ms[-1]), where="post", color=col,
                 lw=1.8 if col == ORANGE else 1.4, label=lab)
    for ax in (ax1, ax2):
        ax.axhline(0, color="k", lw=0.6)
        ax.set_xlim(0, T_END)
        ax.set_xlabel("t (s)")
        ax.legend(fontsize=10, loc="upper right" if ax is ax1 else "lower right")
    ax1.set_ylabel("c(t)")
    ax2.set_ylabel("m(t), held by the ZOH")
    ax1.set_title(f"regulation from x(0) = (1, 0), T = {T:g} s", fontsize=11)
    ax2.set_title("control effort", fontsize=11)
    fig.tight_layout()

    print(f"rho = {rho:.4g}, T = {T:g} s:  K = [{K[0, 0]:.4f}  {K[0, 1]:.4f}],  m(0) = {-K[0, 0]:.4f}")
    print("poles of Phi - Gamma K:", ", ".join(f"{z.real:.4f}{z.imag:+.4f}j" for z in poles),
          f"   |z| = {np.max(np.abs(poles)):.4f}")
    print(f"optimal cost from x(0) = (1, 0): x(0)' P x(0) = {P[0, 0]:.4f}")
    plt.show()


try:
    from ipywidgets import interact, FloatSlider
    interact(explore,
             log10_rho=FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.1,
                                   description="log10 ρ", continuous_update=False),
             T=FloatSlider(value=1.0, min=0.2, max=2.0, step=0.1, description="T (s)",
                           continuous_update=False))
except ImportError:
    explore()

## Hands-on

**Hands-on, Chapter 14.** These cells go with the Hands-on exercises at the end of
Chapter 14. Do each exercise on paper first and write down your prediction; then use the
matching section below to check it. Each section has sliders: change one thing at a time.


In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve_discrete_are

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"

In [ ]:
# ---------------------------------------------------------------- H1
def h1_two_zones(b=0.4, start="imbalance [1, -1]"):
    """H1: one heater, two zones x(k+1) = diag(0.5, b) x + [1, 1]' m; reach x(2) = 0."""
    Phi = np.diag([0.5, b])
    Gam = np.array([1.0, 1.0])
    x0 = np.array([1.0, -1.0]) if start.startswith("imbalance") else np.array([1.0, 1.0])
    # x(2) = Phi^2 x0 + Phi Gam m0 + Gam m1 = 0
    M = np.column_stack([Phi @ Gam, Gam])
    if abs(np.linalg.det(M)) < 1e-9:
        m = np.array([np.nan, np.nan])
    else:
        m = np.linalg.solve(M, -Phi @ Phi @ x0)
    n = 8
    x = np.zeros((n + 1, 2))
    x[0] = x0
    u = np.zeros(n + 1)
    u[:2] = np.nan_to_num(m)
    for k in range(n):
        x[k + 1] = Phi @ x[k] + Gam * u[k]
    k = np.arange(n + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 3.6))
    ax1.plot(k, x[:, 0], "o-", color=ORANGE, ms=5, label="zone 1")
    ax1.plot(k, x[:, 1], "s-", color=BLUE, ms=5, label="zone 2")
    ax1.axhline(0, color="k", lw=0.6)
    ax1.set_xlabel("k")
    ax1.set_ylabel("deviation (deg C)")
    ax1.legend(loc="lower right", fontsize=9)
    ax2.step(k, u, where="post", color=TEAL, lw=1.8)
    ax2.axhline(0, color="k", lw=0.6)
    ax2.set_xlabel("k")
    ax2.set_ylabel("heater m(k) (W)")
    if np.isnan(m[0]):
        fig.suptitle(f"b = {b:.2f}: the two zones cannot be separated (rank 1)", fontsize=11)
    else:
        fig.suptitle(f"b = {b:.2f}, start {x0}: m(0) = {m[0]:.2f} W, m(1) = {m[1]:.2f} W",
                     fontsize=11)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H2
def h2_tank_design(design="classmate"):
    """H2: two tanks, poles 0.2 and 0.4 wanted; step response with the classmate's or the fixed K, N."""
    Phi = np.array([[0.5, 0.0], [0.25, 0.75]])
    Gam = np.array([1.0, 0.0])
    C = np.array([0.0, 1.0])
    K, N = (np.array([0.97, 0.77]), 0.5) if design == "classmate" else (np.array([0.65, 0.77]), 1.92)
    n = 20
    x = np.zeros(2)
    c = np.zeros(n + 1)
    for k in range(n):
        m = -K @ x + N * 1.0
        x = Phi @ x + Gam * m
        c[k + 1] = C @ x
    poles = np.linalg.eigvals(Phi - np.outer(Gam, K))
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(np.arange(n + 1), c, "o-", color=ORANGE, ms=5, label="lower level c(k)")
    ax.axhline(1, color=GRAY, ls="--", lw=1, label="r = 1")
    ax.set_ylim(-0.1, 1.3)
    ax.set_xlabel("k")
    ax.set_title(f"{design}: K = {K}, N = {N}, poles {np.round(np.sort(poles.real), 3)}, "
                 f"final c = {c[-1]:.3f}", fontsize=11)
    ax.legend(loc="lower right", fontsize=9)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H3
def h3_expensive_control(log10_rho=0.0, plant="reactor: x+ = 2x + m"):
    """H3: scalar LQR, J = sum x^2 + rho m^2; closed-loop pole against rho."""
    a = 2.0 if plant.startswith("reactor") else 0.5

    def pole(rho):
        p = solve_discrete_are(np.array([[a]]), np.array([[1.0]]), np.array([[1.0]]),
                               np.array([[rho]]))[0, 0]
        K = a * p / (rho + p)
        return a - K, K, p

    rhos = np.logspace(-2, 4, 200)
    q_curve = [pole(r)[0] for r in rhos]
    rho = 10 ** log10_rho
    q, K, p = pole(rho)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 3.8))
    ax1.semilogx(rhos, q_curve, color=BLUE, lw=1.6)
    ax1.plot(rho, q, "o", color=ORANGE, ms=9)
    ax1.axhline(1 / a if a > 1 else a, color=GRAY, ls="--", lw=1)
    ax1.set_ylim(-0.05, 1.05)
    ax1.set_xlabel("rho")
    ax1.set_ylabel("closed-loop pole")
    k = np.arange(13)
    x = q ** k
    ax2.plot(k, x, "o-", color=ORANGE, ms=5, label="x(k), x(0) = 1")
    ax2.step(k, -K * x, where="post", color=TEAL, lw=1.6, label="m(k)")
    ax2.axhline(0, color="k", lw=0.6)
    ax2.set_xlabel("k")
    ax2.legend(loc="lower right", fontsize=9)
    fig.suptitle(f"{plant}, rho = {rho:.3g}: p = {p:.4g}, K = {K:.4f}, pole = {q:.4f}",
                 fontsize=11)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H4
def h4_velocity_observer(p1=0.3, p2=0.3):
    """H4: conveyor, Phi = [[1, 0.05], [0, 0.5]], c = x1; prediction observer with poles p1, p2."""
    Phi = np.array([[1.0, 0.05], [0.0, 0.5]])
    C = np.array([[1.0, 0.0]])
    Ob = np.vstack([C, C @ Phi])
    alpha = Phi @ Phi - (p1 + p2) * Phi + p1 * p2 * np.eye(2)
    L = alpha @ np.linalg.solve(Ob, np.array([0.0, 1.0]))
    F = Phi - np.outer(L, C)
    n = 25
    e = np.zeros((n + 1, 2))
    e[0] = [0.0, 20.0]                 # velocity guess off by 20 mm/s
    for k in range(n):
        glitch = 1.0 if k == 10 else 0.0   # one encoder reading off by 1 mm
        e[k + 1] = F @ e[k] - L * glitch
    fast = max(abs(p1), abs(p2)) <= 0.36 + 1e-12
    quiet = abs(L[1]) <= 1.25 + 1e-12
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(np.arange(n + 1), e[:, 1], "o-", color=ORANGE, ms=4, label="velocity error (mm/s)")
    ax.axvline(10, color=GRAY, ls=":", lw=1)
    ax.axhspan(-1.25, 1.25, color=BLUE, alpha=0.12, label="glitch budget 1.25 mm/s")
    ax.set_ylim(-6, 21)
    ax.set_xlabel("k")
    ax.legend(loc="upper right", fontsize=9)
    ax.set_title(f"L = [{L[0]:.3f}, {L[1]:.3f}]: fast enough {'yes' if fast else 'no'}, "
                 f"glitch test {'yes' if quiet else 'no'}", fontsize=11)
    plt.show()

In [ ]:
try:
    from ipywidgets import interact, FloatSlider, Dropdown
    interact(h1_two_zones, b=FloatSlider(value=0.4, min=0.0, max=0.5, step=0.01, description="b",
                                         continuous_update=False),
             start=Dropdown(options=["imbalance [1, -1]", "uniform [1, 1]"],
                            value="imbalance [1, -1]", description="start"))
    interact(h2_tank_design, design=Dropdown(options=["classmate", "corrected"],
                                             value="classmate", description="design"))
    interact(h3_expensive_control,
             log10_rho=FloatSlider(value=0.0, min=-2, max=4, step=0.1, description="log10 rho",
                                   continuous_update=False),
             plant=Dropdown(options=["reactor: x+ = 2x + m", "stable: x+ = 0.5x + m"],
                            value="reactor: x+ = 2x + m", description="plant"))
    interact(h4_velocity_observer,
             p1=FloatSlider(value=0.3, min=-0.5, max=0.6, step=0.01, description="p1",
                            continuous_update=False),
             p2=FloatSlider(value=0.3, min=-0.5, max=0.6, step=0.01, description="p2",
                            continuous_update=False))
except ImportError:
    h1_two_zones()
    h2_tank_design()
    h3_expensive_control()
    h4_velocity_observer()